# Pipeline Step 08 — Panel Regression Analysis

Tests whether rising healthcare violence (D1, D2) is associated with declining
TUS preference for high-exposure specialties relative to low-exposure ones.

## Inputs
- `data/processed/tus_panel_macro.csv` — branch × period panel (from step 01)
- `data/processed/gnews_d1_sentiment_final.csv` — D1 treatment variable (from step 04)
- `data/processed/eksi_d2_yearly.csv` — D2 treatment variable (from step 07)

## Design
- **D1** (Google News article count) × exposure group interaction
- **D2** (Eksi Sozluk violence proportion) × exposure group interaction
- Outcome: `score_min` (minimum TUS admission score per branch × period)
- Primary specification: year-level aggregated OLS with quadratic time trend
- Robustness: panel FE (branch fixed effects), year dummy, outlier year exclusion

## Exposure classification
- **High**: patient-facing procedural specialties (ER, surgery, OB-GYN, etc.)
- **Low**: lab/preclinical specialties (radiology, pathology, biochemistry, etc.)
- **Intermediate**: all remaining branches
- **Excluded**: Plastic Surgery (unique high/low ambiguity)

## Notebook sections
1. Shared config + data loader
2. Correlation analysis
3. Summary statistics
4. D1 / D2 trend charts
5. Primary regression — year-level OLS
6. Panel FE regression (robustness)
7. Robustness — log_quota sensitivity
8. Robustness — year dummy vs quadratic trend
9. Robustness — outlier year test

In [ ]:
# !pip install linearmodels statsmodels pandas numpy matplotlib scipy -q

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")

# ── PATH CONFIG ───────────────────────────────────────────────────────────────
TUS_PATH = Path("data/processed/tus_panel_macro.csv")
D1_PATH  = Path("data/processed/gnews_d1_sentiment_final.csv")
D2_PATH  = Path("data/processed/eksi_d2_yearly.csv")
# ─────────────────────────────────────────────────────────────────────────────

# ── BRANCH CLASSIFICATION ─────────────────────────────────────────────────────
# High-exposure: patient-facing specialties with documented violence risk
HIGH_EXPOSURE = [
    "Acil Tıp",
    "Genel Cerrahi",
    "Beyin ve Sinir Cerrahisi",
    "Kalp ve Damar Cerrahisi",
    "Göğüs Cerrahisi",
    "Kadın Hastalıkları ve Doğum",
    "Çocuk Cerrahisi",
    "Üroloji",
    "Ortopedi ve Travmatoloji",
]

# Low-exposure: lab / preclinical specialties with minimal direct patient contact
LOW_EXPOSURE = [
    "Deri ve Zührevi Hastalıkları",
    "Radyoloji",
    "Nükleer Tıp",
    "Tıbbi Patoloji",
    "Tıbbi Biyokimya",
    "Anatomi",
    "Histoloji ve Embriyoloji",
    "Fizyoloji",
    "Tıbbi Farmakoloji",
]

# Excluded: ambiguous classification (cosmetic vs reconstructive)
EXCLUDE = ["Plastik, Rekonstrüktif ve Estetik Cerrahi"]

# Columns carried over from earlier pipeline iterations — drop on load
_STALE_COLS = [
    "d1_count", "d1_binary", "d1_keyword_coverage",
    "n_violence", "violence_ratio", "d1_count_z", "d2_rate_z", "d2_rate",
    "cpi_annual_mean", "usd_try",
]

SEP = "=" * 70

In [ ]:
def _load_panel(
    d2_cols: list[str] | None = None,
) -> pd.DataFrame:
    """
    Loads and joins TUS, D1, and D2 data into a branch × period panel.

    d2_cols: subset of D2 columns to keep (default: all).
    Returns the merged panel with exposure classification,
    log_quota, year_c, year_c2, post_reform, and z-scored treatment vars.
    """
    tus = pd.read_csv(TUS_PATH).dropna(subset=["score_min"])
    d1  = pd.read_csv(D1_PATH)[["year", "d1_count"]]
    d2  = pd.read_csv(D2_PATH)

    # Rename D2 columns to short analysis names
    d2 = d2.rename(columns={
        "d2_violence_proportion": "d2_annot",
        "mean_violence_score":    "d2_severity",
        "anti_violence_rate":     "anti_violence",
    })
    if d2_cols is not None:
        d2 = d2[["year"] + [c for c in d2_cols if c in d2.columns]]

    # Branch classification
    tus = tus[~tus["branch"].isin(EXCLUDE)]
    tus["exposure"] = tus["branch"].apply(
        lambda b: "high" if b in HIGH_EXPOSURE else ("low" if b in LOW_EXPOSURE else "intermediate")
    )

    # Feature engineering
    tus["log_quota"]   = np.log1p(tus["quota_count"])
    tus["year_c"]      = tus["year"] - tus["year"].mean()
    tus["year_c2"]     = tus["year_c"] ** 2
    tus["post_reform"] = (tus["year"] >= 2022).astype(int)
    tus["high_exp"]    = (tus["exposure"] == "high").astype(int)
    tus["mid_exp"]     = (tus["exposure"] == "intermediate").astype(int)

    # Drop stale columns from earlier pipeline runs
    tus = tus.drop(columns=[c for c in _STALE_COLS if c in tus.columns])

    for df in [tus, d1, d2]:
        df["year"] = df["year"].astype(int)

    panel = tus.merge(d1, on="year", how="left").merge(d2, on="year", how="left")

    # Z-score treatment variables
    for col in ["d1_count", "d2_annot", "d2_severity", "anti_violence"]:
        if col in panel.columns:
            panel[f"{col}_z"] = (panel[col] - panel[col].mean()) / panel[col].std()

    # Interaction terms: treatment_z × exposure dummy
    for d in ["d1_count_z", "d2_annot_z", "d2_severity_z"]:
        if d in panel.columns:
            panel[f"{d}_x_high"] = panel[d] * panel["high_exp"]
            panel[f"{d}_x_mid"]  = panel[d] * panel["mid_exp"]

    return panel


def _year_agg(panel: pd.DataFrame) -> pd.DataFrame:
    """Collapses branch-level panel to year × exposure group aggregates."""
    agg_cols = {
        "score_min":      ("score_min",      "mean"),
        "d1_count_z":     ("d1_count_z",     "first"),
        "d2_annot_z":     ("d2_annot_z",     "first"),
        "d2_severity_z":  ("d2_severity_z",  "first"),
        "year_c":         ("year_c",         "first"),
        "year_c2":        ("year_c2",        "first"),
        "post_reform":    ("post_reform",    "first"),
        "high_exp":       ("high_exp",       "first"),
        "mid_exp":        ("mid_exp",        "first"),
    }
    # Only include columns present in the panel
    agg_cols = {k: v for k, v in agg_cols.items() if v[0] in panel.columns}

    agg = panel.groupby(["year", "exposure"]).agg(**agg_cols).reset_index()

    for d in ["d1_count_z", "d2_annot_z", "d2_severity_z"]:
        if d in agg.columns:
            agg[f"{d}_x_high"] = agg[d] * agg["high_exp"]
            agg[f"{d}_x_mid"]  = agg[d] * agg["mid_exp"]

    return agg


def _run_ols(label: str, data: pd.DataFrame, formula: str,
             results: list, cluster_col: str | None = None) -> object:
    """Fits OLS, prints key interaction coefficients, appends to results list."""
    try:
        if cluster_col and cluster_col in data.columns:
            m = smf.ols(formula, data=data).fit(
                cov_type="cluster",
                cov_kwds={"groups": data[cluster_col]},
            )
        else:
            m = smf.ols(formula, data=data).fit()

        print(f"\n  [{label}]  N={int(m.nobs):,}  R²={m.rsquared:.3f}")

        key_vars = [
            "d1_count_z", "d2_annot_z", "d2_severity_z",
            "d1_count_z_x_high", "d1_count_z_x_mid",
            "d2_annot_z_x_high", "d2_annot_z_x_mid",
            "d2_severity_z_x_high", "d2_severity_z_x_mid",
        ]
        row = {"model": label, "n": int(m.nobs), "r2": round(m.rsquared, 3)}
        for var in key_vars:
            if var in m.params:
                p   = m.pvalues[var]
                sig = "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else ""
                print(f"    {var:32s}: coef={m.params[var]:+7.3f}  SE={m.bse[var]:.3f}  p={p:.3f}  {sig}")
                row[f"{var}_coef"] = round(m.params[var], 3)
                row[f"{var}_p"]    = round(p, 3)
                row[f"{var}_sig"]  = sig
        results.append(row)
        return m
    except Exception as e:
        print(f"  [{label}] FAILED: {e}")
        return None


print("Config loaded.")
print(f"  High-exposure branches : {len(HIGH_EXPOSURE)}")
print(f"  Low-exposure branches  : {len(LOW_EXPOSURE)}")
print(f"  Excluded               : {EXCLUDE}")

## 1. Correlation Analysis
Spearman correlations between D1/D2 versions (year-level) and between
each treatment variable and the outcome (branch-level).

In [ ]:
panel = _load_panel()

# ── D1 / D2 inter-measure correlations (year-level, N=13) ────────────────────
annual = panel.drop_duplicates("year")[["year", "d1_count", "d2_annot", "d2_severity"]].sort_values("year")

print(SEP)
print("D1 / D2 inter-measure Spearman correlations (year-level, N=13)")
print(SEP)
pairs = [
    ("d1_count",   "d2_annot",    "D1 count   vs D2 annot"),
    ("d1_count",   "d2_severity", "D1 count   vs D2 severity"),
    ("d2_annot",   "d2_severity", "D2 annot   vs D2 severity"),
]
for xa, xb, label in pairs:
    r, p = spearmanr(annual[xa], annual[xb])
    print(f"  {label}: r={r:+.3f}  p={p:.3f}")

# ── Spearman by exposure group ────────────────────────────────────────────────
print(f"\n{SEP}")
print("Spearman: treatment vs score_min by exposure group (branch-level)")
print(SEP)
for group in ["high", "intermediate", "low"]:
    sub = panel[panel["exposure"] == group]
    for tv, label in [("d1_count", "D1"), ("d2_annot", "D2 annot")]:
        r, p = spearmanr(sub[tv], sub["score_min"])
        print(f"  [{group:12s}] {label}: r={r:+.3f}  p={p:.3f}  (n={len(sub):,})")

## 2. Summary Statistics

In [ ]:
panel = _load_panel()

print("=" * 72)
print("TABLE 1 — Branch-Level Panel")
print("=" * 72)
print(f"\n{'Variable':<35} {'N':>7} {'Mean':>9} {'SD':>9} {'Min':>9} {'Max':>9}")
print("-" * 72)

panel_vars = {
    "score_min":   "Minimum TUS score (outcome)",
    "quota_count": "Branch quota count",
    "log_quota":   "Log quota count",
    "d1_count":    "D1: Google News article count",
    "d2_annot":    "D2: Violence flag rate (annotation)",
    "d2_severity": "D2: Mean violence score (severity)",
}
for col, label in panel_vars.items():
    s = panel[col].dropna()
    print(f"  {label:<33} {len(s):>7,} {s.mean():>9.3f} {s.std():>9.3f} {s.min():>9.3f} {s.max():>9.3f}")

print("\n  — score_min by exposure group:")
for group in ["high", "intermediate", "low"]:
    s = panel[panel["exposure"] == group]["score_min"].dropna()
    print(f"    {group:12s}: n={len(s):,}  mean={s.mean():.1f}  sd={s.std():.1f}")

print(f"\nTotal panel obs: {len(panel):,}")

## 3. D1 / D2 Annual Trend Charts

In [ ]:
d1 = pd.read_csv(D1_PATH)[["year", "d1_count"]]
d2 = pd.read_csv(D2_PATH)[["year", "d2_violence_proportion", "mean_violence_score"]].rename(
    columns={"d2_violence_proportion": "d2_annot", "mean_violence_score": "d2_severity"}
)
d1["year"] = d1["year"].astype(int)
d2["year"] = d2["year"].astype(int)
annual = d1.merge(d2, on="year")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Treatment Variables — Annual Trend (2013–2025)",
             fontsize=13, fontweight="bold", y=1.01)

ax1.plot(annual["year"], annual["d1_count"],
         color="#2E75B6", marker="o", linewidth=2.5, markersize=6)
ax1.fill_between(annual["year"], annual["d1_count"], alpha=0.1, color="#2E75B6")
ax1.axvline(x=2022, color="red", linestyle="--", alpha=0.7, label="2022 Karakaya event")
peak_year = annual.loc[annual["d1_count"].idxmax(), "year"]
peak_val  = annual["d1_count"].max()
ax1.annotate(f"Peak: {peak_val}",
             xy=(peak_year, peak_val),
             xytext=(peak_year - 2, peak_val - 30),
             arrowprops=dict(arrowstyle="->", color="red", alpha=0.7),
             fontsize=9, color="red")
ax1.set_title("D1 — Google News Article Count", fontsize=11)
ax1.set_xlabel("Year")
ax1.set_ylabel("Annual article count")
ax1.legend(fontsize=9)
ax1.grid(axis="y", alpha=0.3)
ax1.xaxis.set_major_locator(ticker.MultipleLocator(2))
ax1.set_ylim(bottom=0)

ax2.plot(annual["year"], annual["d2_annot"],
         color="#2E75B6", marker="o", linewidth=2.5, markersize=6,
         label="D2 annot (flag rate)")
ax2.plot(annual["year"], annual["d2_severity"],
         color="#70AD47", marker="s", linewidth=2, markersize=6,
         linestyle="--", label="D2 severity (mean score)")
ax2.axvline(x=2022, color="red", linestyle="--", alpha=0.7, label="2022 event")
ax2.set_title("D2 — Eksi Sozluk Violence Measures", fontsize=11)
ax2.set_xlabel("Year")
ax2.set_ylabel("Violence proportion / mean score")
ax2.legend(fontsize=9)
ax2.grid(axis="y", alpha=0.3)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(2))
ax2.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig("d1_d2_trends.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: d1_d2_trends.png")

print("\nAnnual D1/D2 values:")
print(annual[["year", "d1_count", "d2_annot", "d2_severity"]].to_string(index=False))

## 4. Primary Regression — Year-Level Aggregated OLS

Year × exposure group aggregates (N=39 for 13 years × 3 groups).
Plain OLS — clustered SE not applicable (only 3 clusters).

| Model | Spec |
|---|---|
| A1 | D1 only + quadratic trend (primary) |
| A2 | D2 annot only + quadratic trend |
| A3 | D1 + D2 together |
| A4 | D1 only, no post_reform dummy (sensitivity) |
| A5 | D2 severity (robustness vs D2 annot) |

In [ ]:
panel   = _load_panel()
agg     = _year_agg(panel)
results = []

print(SEP)
print("PRIMARY — YEAR-LEVEL AGGREGATED OLS (N=39)")
print("Plain OLS — 3 clusters too few for clustered SE")
print(SEP)

# A1: D1 only (primary — cleanest spec)
_run_ols(
    "A1 D1 only | quadratic", agg, results=results,
    formula="score_min ~ d1_count_z + d1_count_z_x_high + d1_count_z_x_mid"
            " + high_exp + mid_exp + year_c + year_c2 + post_reform",
)

# A2: D2 annot only
_run_ols(
    "A2 D2 annot only | quadratic", agg, results=results,
    formula="score_min ~ d2_annot_z + d2_annot_z_x_high + d2_annot_z_x_mid"
            " + high_exp + mid_exp + year_c + year_c2 + post_reform",
)

# A3: D1 + D2 together
_run_ols(
    "A3 D1 + D2 annot | quadratic", agg, results=results,
    formula="score_min ~ d1_count_z + d2_annot_z"
            " + d1_count_z_x_high + d1_count_z_x_mid"
            " + d2_annot_z_x_high + d2_annot_z_x_mid"
            " + high_exp + mid_exp + year_c + year_c2 + post_reform",
)

# A4: D1 only, no post_reform (sensitivity to reform dummy)
_run_ols(
    "A4 D1 only | quadratic, no post_reform", agg, results=results,
    formula="score_min ~ d1_count_z + d1_count_z_x_high + d1_count_z_x_mid"
            " + high_exp + mid_exp + year_c + year_c2",
)

# A5: D2 severity (alternative D2 measure)
_run_ols(
    "A5 D2 severity | quadratic", agg, results=results,
    formula="score_min ~ d2_severity_z + d2_severity_z_x_high + d2_severity_z_x_mid"
            " + high_exp + mid_exp + year_c + year_c2 + post_reform",
)

pd.DataFrame(results).to_csv("regression_results.csv", index=False)
print(f"\n\nSaved: regression_results.csv ({len(results)} models)")

print(f"\n{SEP}")
print("Interpretation guide:")
print("  A1 d1_count_z_x_high negative and significant → H1 supported")
print("  A1 vs A4: does post_reform dummy absorb the effect?")
print("  A1 vs A3: does D2 attenuate D1 when both entered?")
print("  A2 vs A5: d2_annot vs d2_severity consistency")
print(SEP)

## 5. Panel FE Regression (Robustness)

Branch fixed effects via `linearmodels.PanelOLS`.
Entity: branch; Time: period_id (year × period).
Clustered SE by branch.

| Model | Spec |
|---|---|
| B1 | D1 only + log_quota + FE |
| B2 | D2 annot only + log_quota + FE |
| B3 | D1 + D2 + log_quota + FE |

In [ ]:
from linearmodels.panel import PanelOLS

panel = _load_panel()
panel["period_id"] = panel["year"] * 10 + panel["period"]
fe_panel = panel.set_index(["branch", "period_id"])

def _run_panel_fe(label: str, exog_cols: list[str]) -> None:
    print(f"\n{SEP}")
    print(f"MODEL: {label}")
    print(SEP)
    sub  = fe_panel[["score_min"] + exog_cols].dropna()
    exog = sm.add_constant(sub[exog_cols])
    mod  = PanelOLS(
        dependent=sub["score_min"],
        exog=exog,
        entity_effects=True,
        time_effects=False,
    )
    res = mod.fit(cov_type="clustered", cluster_entity=True)
    print(res.summary.tables[1])

# B1: D1 only
_run_panel_fe(
    "B1 D1 only | branch FE + log_quota",
    ["d1_count_z", "d1_count_z_x_high", "d1_count_z_x_mid",
     "log_quota", "year_c", "year_c2"],
)

# B2: D2 annot only
_run_panel_fe(
    "B2 D2 annot only | branch FE + log_quota",
    ["d2_annot_z", "d2_annot_z_x_high", "d2_annot_z_x_mid",
     "log_quota", "year_c", "year_c2"],
)

# B3: D1 + D2 together
_run_panel_fe(
    "B3 D1 + D2 annot | branch FE + log_quota",
    ["d1_count_z", "d2_annot_z",
     "d1_count_z_x_high", "d1_count_z_x_mid",
     "d2_annot_z_x_high", "d2_annot_z_x_mid",
     "log_quota", "year_c", "year_c2"],
)

## 6. Robustness — log_quota Sensitivity

Tests whether including log_quota attenuates the D1 coefficient.
D1 and log_quota are highly correlated (r ≈ +0.99 at the year level),
so this check verifies that the interaction effect is not absorbed by the control.

In [ ]:
from linearmodels.panel import PanelOLS

panel = _load_panel()
panel["period_id"] = panel["year"] * 10 + panel["period"]
fe_panel = panel.set_index(["branch", "period_id"])

def _run_fe_spec(label: str, exog_cols: list[str]) -> None:
    print(f"\n{SEP}\n{label}\n{SEP}")
    sub  = fe_panel[["score_min"] + exog_cols].dropna()
    exog = sm.add_constant(sub[exog_cols])
    res  = PanelOLS(
        dependent=sub["score_min"],
        exog=exog,
        entity_effects=True,
    ).fit(cov_type="clustered", cluster_entity=True)
    print(res.summary.tables[1])

base_d1 = ["d1_count_z", "d1_count_z_x_high", "d1_count_z_x_mid", "year_c", "year_c2"]
base_d2 = ["d2_annot_z", "d2_annot_z_x_high", "d2_annot_z_x_mid", "year_c", "year_c2"]

_run_fe_spec("D1 only — without log_quota",            base_d1)
_run_fe_spec("D1 only — with log_quota",               base_d1 + ["log_quota"])
_run_fe_spec("D2 annot only — without log_quota",      base_d2)
_run_fe_spec("D2 annot only — with log_quota",         base_d2 + ["log_quota"])

## 7. Robustness — Year Dummy vs Quadratic Trend

Year dummies absorb D1/D2 in the panel (perfect collinearity),
so this test is only run on the year-level aggregate OLS.

In [ ]:
panel   = _load_panel()
agg     = _year_agg(panel)
results = []

print(SEP)
print("YEAR DUMMY TEST — year-level OLS")
print("Year dummies absorb D1/D2 in panel FE (singular matrix).")
print(SEP)

# Quadratic trend (primary)
_run_ols(
    "D1 | quadratic trend", agg, results=results,
    formula="score_min ~ d1_count_z + d1_count_z_x_high + d1_count_z_x_mid"
            " + high_exp + mid_exp + year_c + year_c2 + post_reform",
)

# Year dummy (year-level only)
_run_ols(
    "D1 | year dummy", agg, results=results,
    formula="score_min ~ d1_count_z + d1_count_z_x_high + d1_count_z_x_mid"
            " + high_exp + mid_exp + C(year)",
)

_run_ols(
    "D2 annot | quadratic trend", agg, results=results,
    formula="score_min ~ d2_annot_z + d2_annot_z_x_high + d2_annot_z_x_mid"
            " + high_exp + mid_exp + year_c + year_c2 + post_reform",
)

_run_ols(
    "D2 annot | year dummy", agg, results=results,
    formula="score_min ~ d2_annot_z + d2_annot_z_x_high + d2_annot_z_x_mid"
            " + high_exp + mid_exp + C(year)",
)

## 8. Robustness — Outlier Year Test

Tests whether the interaction sign change is driven by a single outlier year.
Steps:
1. IQR and z-score outlier detection on D1 (year-level)
2. Leave-one-year-out: refit A1 excluding each year, check interaction stability

In [ ]:
panel = _load_panel()
agg   = _year_agg(panel)

# ── Part 1: D1 outlier detection (year-level, N=13) ──────────────────────────
annual_d1 = panel.drop_duplicates("year")[["year", "d1_count"]].sort_values("year")

mean_d1 = annual_d1["d1_count"].mean()
std_d1  = annual_d1["d1_count"].std()
q1      = annual_d1["d1_count"].quantile(0.25)
q3      = annual_d1["d1_count"].quantile(0.75)
iqr     = q3 - q1

annual_d1["z_score"]     = (annual_d1["d1_count"] - mean_d1) / std_d1
annual_d1["iqr_outlier"] = ((annual_d1["d1_count"] < q1 - 1.5 * iqr) |
                             (annual_d1["d1_count"] > q3 + 1.5 * iqr))
annual_d1["z_outlier"]   = annual_d1["z_score"].abs() > 2

print(SEP)
print("D1 Outlier Detection (year-level)")
print(SEP)
print(annual_d1.to_string(index=False))

# ── Part 2: Leave-one-year-out interaction stability ────────────────────────
print(f"\n{SEP}")
print("Leave-one-year-out: A1 interaction coefficient stability")
print(f"{'Year excl.':>12} {'d1_z_x_high coef':>18} {'p':>8} {'d1_z_x_mid coef':>18} {'p':>8}")
print("-" * 70)

formula = ("score_min ~ d1_count_z + d1_count_z_x_high + d1_count_z_x_mid"
           " + high_exp + mid_exp + year_c + year_c2 + post_reform")

for yr in sorted(agg["year"].unique()):
    sub = agg[agg["year"] != yr]
    try:
        m = smf.ols(formula, data=sub).fit()
        c_high = m.params.get("d1_count_z_x_high", float("nan"))
        p_high = m.pvalues.get("d1_count_z_x_high", float("nan"))
        c_mid  = m.params.get("d1_count_z_x_mid",  float("nan"))
        p_mid  = m.pvalues.get("d1_count_z_x_mid",  float("nan"))
        print(f"  excl. {yr:4d}  {c_high:>+16.3f}  {p_high:>8.3f}  {c_mid:>+16.3f}  {p_mid:>8.3f}")
    except Exception as e:
        print(f"  excl. {yr:4d}  FAILED: {e}")